# Raw Audit
## Check and record the problems for raw job information from Jsearch.

In [1]:
import sys
sys.path.append("..")
import pandas as pd
from IPython.display import display
from config.settings import RAW_DATA_DIR, OUTPUT, SENIORITY_EXCLUDE_KEYWORDS
import re

In [2]:
# read file
df_1 = pd.read_json(RAW_DATA_DIR / 'jobs_20260911_071658.jsonl', lines=True)
df_2 = pd.read_json(RAW_DATA_DIR / 'jobs_20260920_005742.jsonl', lines=True)
df_3 = pd.read_json(RAW_DATA_DIR / 'jobs_20260920_011920.jsonl', lines=True)
df_4 = pd.read_json(RAW_DATA_DIR / 'jobs_20260920_012618.jsonl', lines=True)
pilot_ids = set(df_1['job_id'].dropna())
new_jobs = df_2.loc[~df_2['job_id'].isin(pilot_ids)].copy()

print('Rows in df_2:', len(df_2))
print('Duplicated jobs with pilot_ids:', len(df_2) - len(new_jobs))
print('New jobs:', len(new_jobs))

df_raw = pd.concat([df_1, df_2, df_3, df_4],ignore_index=True)

Rows in df_2: 83
Duplicated jobs with pilot_ids: 0
New jobs: 83


In [3]:
# Check the basic structure:
display(df_raw.info())
display(df_raw.head())
display(df_raw.columns)
display(df_raw.shape)

<class 'pandas.DataFrame'>
RangeIndex: 643 entries, 0 to 642
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   job_id                      643 non-null    str    
 1   job_title                   643 non-null    str    
 2   employer_name               643 non-null    str    
 3   job_state                   497 non-null    str    
 4   job_country                 637 non-null    str    
 5   job_city                    425 non-null    str    
 6   job_latitude                187 non-null    float64
 7   job_longitude               187 non-null    float64
 8   job_employment_type         607 non-null    str    
 9   job_is_remote               643 non-null    bool   
 10  job_posted_at_datetime_utc  631 non-null    str    
 11  job_description             643 non-null    str    
 12  job_apply_link              643 non-null    str    
 13  job_min_salary              0 non-null      fl

None

,job_id,job_title,employer_name,job_state,job_country,job_city,job_latitude,job_longitude,job_employment_type,job_is_remote,...,job_min_salary,job_max_salary,job_salary_currency,job_salary_period,role_family,role_query,search_level,search_query,page_number,collected_at_utc
0,YmluZzotOTk2MDUzMjU0LlJldHJv,Data Analyst Intern,BNP Paribas,QC,CA,Montreal,NaN,NaN,NaN,False,...,NaN,NaN,NaN,NaN,data_analyst,data analyst,internship,"(data analyst) AND (""internship"" OR ""intern"" O...",1,2026-09-11T07:14:18.441029+00:00
1,YmluZzoxMjAxNDU0MTIxLlJldHJv,Data Analyst I – 4-Month Internship/Coop (Wint...,Intact,QC,CA,Montreal,NaN,NaN,NaN,False,...,NaN,NaN,NaN,NaN,data_analyst,data analyst,internship,"(data analyst) AND (""internship"" OR ""intern"" O...",1,2026-09-11T07:14:18.441029+00:00
2,YmluZzotMTc2MjY4OTE5OC5SZXRybw,Data Analyst I - 4-Month Internship/Coop (Wint...,intact financial,QC,CA,Montreal,NaN,NaN,Full-time,False,...,NaN,NaN,NaN,NaN,data_analyst,data analyst,internship,"(data analyst) AND (""internship"" OR ""intern"" O...",1,2026-09-11T07:14:18.441029+00:00
3,YmluZzotODAzMzI2Njk0LlJldHJv,"Intern, GIS Data Analyst",Pomerleau,QC,CA,Montreal,NaN,NaN,NaN,False,...,NaN,NaN,NaN,NaN,data_analyst,data analyst,internship,"(data analyst) AND (""internship"" OR ""intern"" O...",1,2026-09-11T07:14:18.441029+00:00
4,YmluZzotMTE0ODkwODg5My5SZXRybw,"Intern, HR Data Analyst",Pomerleau,QC,CA,Montreal,NaN,NaN,NaN,False,...,NaN,NaN,NaN,NaN,data_analyst,data analyst,internship,"(data analyst) AND (""internship"" OR ""intern"" O...",1,2026-09-11T07:14:18.441029+00:00


Index(['job_id', 'job_title', 'employer_name', 'job_state', 'job_country',
       'job_city', 'job_latitude', 'job_longitude', 'job_employment_type',
       'job_is_remote', 'job_posted_at_datetime_utc', 'job_description',
       'job_apply_link', 'job_min_salary', 'job_max_salary',
       'job_salary_currency', 'job_salary_period', 'role_family', 'role_query',
       'search_level', 'search_query', 'page_number', 'collected_at_utc'],
      dtype='str')

(643, 23)

In [4]:
# Check if all the fields are complete
expected_columns = set(OUTPUT)
actual_columns = set(df_raw.columns)
missing_columns = expected_columns - actual_columns
unexpected_columns = actual_columns - expected_columns
print('Missing columns:', missing_columns)
print('Unexpected columns:', unexpected_columns)

Missing columns: set()
Unexpected columns: set()


In [5]:
# Check missing values and proportion of missing values
missing_report = pd.DataFrame({
    'missing_count': df_raw.isna().sum(),
    'missing_rate': (df_raw.isna().mean() * 100).round(0),
})
display(missing_report.sort_values('missing_rate', ascending=False))

,missing_count,missing_rate
job_salary_period,643,100.0
job_min_salary,643,100.0
job_salary_currency,643,100.0
job_max_salary,643,100.0
job_latitude,456,71.0
job_longitude,456,71.0
job_city,218,34.0
job_state,146,23.0
job_employment_type,36,6.0
job_posted_at_datetime_utc,12,2.0


In [6]:
# Check empty string
text_columns = df_raw.select_dtypes(include = 'str').columns
blank_counts = {
    column: df_raw[column].str.strip().eq('').sum()
    for column in text_columns
}
blank_report = pd.DataFrame({
    'column': blank_counts.keys(),
    'blank_count': blank_counts.values(),
})
display(blank_report)
display(text_columns)

,column,blank_count
0,job_id,0
1,job_title,0
2,employer_name,0
3,job_state,0
4,job_country,0
5,job_city,0
6,job_employment_type,0
7,job_posted_at_datetime_utc,0
8,job_description,0
9,job_apply_link,0


Index(['job_id', 'job_title', 'employer_name', 'job_state', 'job_country',
       'job_city', 'job_employment_type', 'job_posted_at_datetime_utc',
       'job_description', 'job_apply_link', 'role_family', 'role_query',
       'search_level', 'search_query', 'collected_at_utc'],
      dtype='str')

In [7]:
# Check the duplicated job ids
duplicate_count = df_raw.duplicated('job_id').sum()
unique_job_id = df_raw['job_id'].nunique()
print(f'duplicate count: {duplicate_count}')
print(f'unique job id: {unique_job_id}')
duplicated_row = df_raw.loc[df_raw.duplicated('job_id', keep=False), ['job_id', 'job_title', 'employer_name', 'role_family', 'search_level', 'search_query', 'page_number']].sort_values('job_id')
display(duplicated_row.head(10))


duplicate count: 143
unique job id: 500


,job_id,job_title,employer_name,role_family,search_level,search_query,page_number
424,YmluZzo1MjIzNzEyNi5SZXRybw,Data Engineer,TES The Employment Solution Inc.,data_engineer,entry_level,"(""data engineer"" OR ""etl developer"") AND (""ent...",1
365,YmluZzo1MjIzNzEyNi5SZXRybw,Data Engineer,TES The Employment Solution Inc.,data_engineer,internship,"(""data engineer"" OR ""etl developer"") AND (""int...",1
153,YmluZzo1Nzc5NTA0MjMuUmV0cm8,Business Analyst,HollisWealth,business_analyst,entry_level,"(business analyst) AND (""entry level"" OR ""juni...",1
94,YmluZzo1Nzc5NTA0MjMuUmV0cm8,Business Analyst,HollisWealth,business_analyst,internship,"(business analyst) AND (""internship"" OR ""inter...",1
265,YmluZzo1ODAxNTQxMC5SZXRybw,Data & Reporting Analyst (Power BI),Insight Global,bi_analyst,entry_level,"(""bi analyst"" OR ""business intelligence analys...",3
234,YmluZzo1ODAxNTQxMC5SZXRybw,Data & Reporting Analyst (Power BI),Insight Global,bi_analyst,new_grad,"(""bi analyst"" OR ""business intelligence analys...",3
394,YmluZzo2MDY4Njg2LlJldHJv,Data Engineer Cloud AWS,Astek,data_engineer,new_grad,"(""data engineer"" OR ""etl developer"") AND (""new...",1
436,YmluZzo2MDY4Njg2LlJldHJv,Data Engineer Cloud AWS,Astek,data_engineer,entry_level,"(""data engineer"" OR ""etl developer"") AND (""ent...",2
13,YmluZzo2MjMzNjQyMjMuUmV0cm8,National Real Estate Group Intern / Co-Op (Win...,TD,data_analyst,internship,"(data analyst) AND (""internship"" OR ""intern"" O...",2
104,YmluZzo2MjMzNjQyMjMuUmV0cm8,National Real Estate Group Intern / Co-Op (Win...,TD,business_analyst,internship,"(business analyst) AND (""internship"" OR ""inter...",2


In [8]:
# Check the performance of each query
display(df_raw.value_counts('role_family'))
display(df_raw.value_counts('search_level'))
role_level_counts = pd.crosstab(df_raw['role_family'], df_raw['search_level'])
display(role_level_counts)
query_page_counts = df_raw.groupby(['role_family', 'search_level', 'page_number'], as_index=False).size()
display(query_page_counts.head(10))

role_family
data_scientist      150
data_analyst        145
data_engineer       129
business_analyst    123
bi_analyst           96
Name: count, dtype: int64

search_level
internship     266
entry_level    219
new_grad       158
Name: count, dtype: int64

search_level,entry_level,internship,new_grad
role_family,,,
bi_analyst,30,36,30
business_analyst,30,60,33
data_analyst,60,50,35
data_engineer,39,60,30
data_scientist,60,60,30


,role_family,search_level,page_number,size
0,bi_analyst,entry_level,1,10
1,bi_analyst,entry_level,2,10
2,bi_analyst,entry_level,3,10
3,bi_analyst,internship,1,10
4,bi_analyst,internship,2,10
5,bi_analyst,internship,3,10
6,bi_analyst,internship,4,6
7,bi_analyst,new_grad,1,10
8,bi_analyst,new_grad,2,10
9,bi_analyst,new_grad,3,10


In [9]:
# Check the position relevance families(sample survey)
for role_family, group in df_raw.groupby('role_family'):
    print(f'role family: {role_family}')
    sample_size = min(10, len(group))
    sample = group.sample(
        n=sample_size,
        random_state=42,
    )
    display(sample[['job_title', 'employer_name', 'role_family', 'search_level']])

role family: bi_analyst


,job_title,employer_name,role_family,search_level
260,Junior Business Analyst,Insight Global,bi_analyst,entry_level
257,Threat Intelligence Analyst,Alignerr,bi_analyst,entry_level
253,Business Analyst,Mindlance,bi_analyst,entry_level
492,"Climate Reporting Analyst Intern, Environment,...",Concert Properties Ltd.,bi_analyst,internship
213,Business Intelligence Analyst,ROBIC,bi_analyst,new_grad
259,Business Analyst (Junior / Intermediate),CGI,bi_analyst,entry_level
249,BI Analyst/Developer – Compensation Reporting ...,Morgan Stanley,bi_analyst,entry_level
222,"Analyste, Intelligence D'affaires (BI)",Bombardier,bi_analyst,new_grad
180,"Analyst, Business Intelligence (BI)",Bombardier,bi_analyst,internship
190,Senior Data Analyst - Business Intelligence,UAP Inc.,bi_analyst,internship


role family: business_analyst


,job_title,employer_name,role_family,search_level
108,Business System Analyst Co-Op / Intern (Winter...,TD,business_analyst,internship
135,"Senior Business Analyst, New Verticals (Bangko...",Agoda,business_analyst,new_grad
137,ACCELERATE Commercial Banking New Graduate Pro...,Scotiabank,business_analyst,new_grad
179,Experienced Business Analyst Needed,Randstad Canada,business_analyst,entry_level
94,Business Analyst,HollisWealth,business_analyst,internship
130,IT Business Analyst– 4-Month Internship/Coop (...,Intact,business_analyst,new_grad
152,Business Analyst,Industrial Alliance Pacific,business_analyst,entry_level
485,New Grad - Business Analysis & Quality Assuran...,BDO Canada LLP,business_analyst,new_grad
121,PLM Business Analyst,Scalian,business_analyst,new_grad
145,Senior Business Analyst,Joblinxsapp,business_analyst,new_grad


role family: data_analyst


,job_title,employer_name,role_family,search_level
69,Data Analyst Intern,BNP Paribas,data_analyst,entry_level
598,"Remote Online Data Analyst — Part-Time, Entry-...",TELUS Digital AI Data Solutions,data_analyst,entry_level
27,Remote Data Science Intern — GIS & Remote Sensing,DiscoverEE Inc.,data_analyst,internship
19,"Data Science Risk Analyst, Winter 2027 (Co-Op/...",BMO Capital Markets,data_analyst,internship
42,Data Security & DLP Analyst,Alignerr,data_analyst,new_grad
535,Data Scientist / AI,EIC - Experience Internship Canada,data_analyst,internship
544,Spreadsheet Analyst - Fully Remote | Upto $70/hr,Mercor,data_analyst,entry_level
468,Online Data Analyst | Freelance Remote,TELUS Digital AI Data Solutions,data_analyst,entry_level
84,Data Entry Work - Remote ( Wfh ),Only Data Entry,data_analyst,entry_level
18,TD Asset Management Portfolio Analytics Data A...,TD,data_analyst,internship


role family: data_engineer


,job_title,employer_name,role_family,search_level
415,Software Engineer 1,L3HHCM20,data_engineer,new_grad
400,Data Scientist I – 4-Month Internship/Co-Op (W...,Intact,data_engineer,new_grad
379,"Data Analytics Intern, Winter 2027 (Co-Op/Inte...",Randstad Canada,data_engineer,internship
391,Data Engineer,galasolutions,data_engineer,new_grad
588,"Co-op Data Engineer - Vehicle Data, CI​/CD & C...",Jobtailor,data_engineer,internship
416,Manufacturing Design Engineer - Remote,YO IT Consulting,data_engineer,new_grad
429,Data Engineer,Epik Solutions,data_engineer,entry_level
529,Junior Data Engineer: Paid Training to AI/Data...,Jarvis Consulting Group,data_engineer,entry_level
441,Remote Senior Data Engineer: Data & ML Platfor...,Hive,data_engineer,entry_level
386,Data Analyst Intern/Co-Op (Winter 2027),TD,data_engineer,internship


role family: data_scientist


,job_title,employer_name,role_family,search_level
343,Data Scientist II,Intact,data_scientist,entry_level
288,Data Analytics & Governance Intern — 4‑Month C...,ODAIA,data_scientist,internship
571,"Winter 2027 Intern, Data Engineering - Apply",kodiak,data_scientist,internship
348,Data Scientist II,Coursera,data_scientist,entry_level
346,Senior Data Scientist,Giesecke+Devrient,data_scientist,entry_level
301,Data Scientist,Apptad,data_scientist,new_grad
334,Data Scientist,BNP Paribas,data_scientist,entry_level
624,Specialist - Data Sciences,LTM,data_scientist,entry_level
338,Data Scientist,Open Systems Technologies,data_scientist,entry_level
352,Data Scientist II,TD,data_scientist,entry_level


In [10]:
# Initial marking of senior positions
senior_pattern = r'\b(?:' + '|'.join(re.escape(keyword) for keyword in SENIORITY_EXCLUDE_KEYWORDS) + r')\b'
senior_mask = df_raw['job_title'].str.contains(senior_pattern, case=False, na=False)
print('Potential senior jobs:', senior_mask.sum())
print(f'Potential senior rate: {(senior_mask.mean()*100).round(2)}%')
senior_rows = df_raw.loc[senior_mask, ['job_title', 'employer_name', 'role_family', 'search_level']]
display(senior_rows)

Potential senior jobs: 79
Potential senior rate: 12.29%


,job_title,employer_name,role_family,search_level
43,Senior Data Analyst I,Mistplay,data_analyst,new_grad
44,"Senior Data Analyst I, Product",Mistplay,data_analyst,new_grad
45,"Medical Insights, Data Analyst Senior Manager",Pfizer,data_analyst,new_grad
58,"Senior Data Analyst, Business Intelligence",Doane Grant Thornton LLP,data_analyst,new_grad
59,Senior Data Engineer (Analyst/Scientist) - Remote,KAKE,data_analyst,new_grad
...,...,...,...,...
578,Principal Clinical Data Science Programmer,PRA Health Sciences,data_scientist,entry_level
580,Lead Data Scientist​ - R&D at Mastercard Canada,Mastercard,data_scientist,entry_level
582,Sr. Data Scientist (Remote),CrowdStrike,data_scientist,entry_level
629,Principal Clinical Data Science Programmer,PRA Health Sciences,data_scientist,entry_level


In [11]:
# Check the quality for job description.
description_length = df_raw['job_description'].fillna('').str.len()
display(description_length.describe().reset_index(name='description_length'))
short_jobs = df_raw.loc[description_length < 500, ['job_title', 'employer_name', 'job_description']].reset_index(drop=True)
display(short_jobs)

,index,description_length
0,count,643.000000
1,mean,5220.629860
2,std,3223.232172
3,min,203.000000
4,25%,2695.500000
5,50%,5354.000000
6,75%,7099.000000
7,max,18312.000000


,job_title,employer_name,job_description
0,Remote Data Analyst For Latvian Speakers,TELUS Digital AI Data Solutions,A global technology firm based in Canada is se...
1,"Remote Lithuanian Data Analyst (Freelance, Par...",TELUS Digital AI Data Solutions,A tech company specializing in AI solutions is...
2,Remote Data Entry Operator (Junior) - Growth Path,Only Data Entry,A data-focused company is seeking a Junior Dat...
3,Remote Data Entry Coordinator - Entry Level,Only Data Entry,A data management company is seeking a Remote ...
4,Procurement & Business Analyst Intern — Proces...,RBC,RBC in Ontario is offering a 4-month Winter 20...
5,AI Project Business Analyst - New Grad,UniSysTech Consulting,A consulting company in Toronto is seeking a n...
6,"Finance BI Developer, Finance Business Intelli...",Autodesk,"• Develop and maintain Dashboards, Apps and re..."
7,"Remote Data Engineer, BI & Reporting Analytic...",Veem,"Veem Inc. is seeking a Data Engineer, BI & Rep..."
8,Senior Cloud Data Engineer - Platform Moderniz...,Jobgether,Jobgether is seeking a Senior Data Engineer (C...
9,SCALA Data Engineer,"ITMC Systems, Inc",Role: SCALA Data engineer\n\r\nLocation: Montr...


In [12]:
# Date check
posted_at = pd.to_datetime(df_raw['job_posted_at_datetime_utc'], errors='coerce', utc=True)
collected_at = pd.to_datetime(df_raw['collected_at_utc'], errors='coerce', utc=True)
print('Posted cannot change to datetime:', posted_at.isna().sum())
print('Collected cannot change to datetime:', collected_at.isna().sum())
print('Earliest posted time:', posted_at.min())
print('Latest posted time:', posted_at.max())
print('Earliest collected time:', collected_at.min())
print('Latest collected time:', collected_at.max())
posting_days = (collected_at - posted_at).dt.total_seconds()/86400
display(posting_days.describe().reset_index(name='posting_days'))
print('Future-dated postings:', (posting_days < 0).sum())
print('Older than 35 days:', (posting_days > 35).sum())

Posted cannot change to datetime: 12
Collected cannot change to datetime: 0
Earliest posted time: 2026-02-09 16:00:00+00:00
Latest posted time: 2026-09-19 06:00:00+00:00
Earliest collected time: 2026-09-11 07:14:18.441029+00:00
Latest collected time: 2026-09-20 01:26:17.149820+00:00


,index,posting_days
0,count,631.000000
1,mean,13.840042
2,std,16.800406
3,min,0.788954
4,25%,2.859599
5,50%,7.302831
6,75%,19.302567
7,max,213.635322


Future-dated postings: 0
Older than 35 days: 49


In [13]:
# Check the Country, province, city
check_country = df_raw['job_country'].value_counts(dropna=False).reset_index(name='number')
display(check_country)
check_province = df_raw['job_state'].value_counts(dropna=False).reset_index(name='number')
display(check_province)
check_city = df_raw['job_city'].value_counts(dropna=False).reset_index(name='number')
display(check_city)
print('Number of job in California:', df_raw['job_state'].eq('California').sum())


,job_country,number
0,CA,637
1,NaN,6


,job_state,number
0,QC,309
1,NaN,146
2,ON,70
3,ONTARIO,43
4,Ontario,30
5,California,19
6,Saskatchewan,11
7,British Columbia,5
8,BC,4
9,Quebec,4


,job_city,number
0,Montreal,273
1,NaN,218
2,Toronto,73
3,Toronto On,23
4,Markham,10
5,Cañada,7
6,Montréal,6
7,Mississauga,5
8,Québec City,4
9,Vancouver,3


Number of job in California: 19


In [14]:
# Check the limitation for Montreal
job_distribution = df_raw.groupby('job_city', dropna=False).agg(
    raw_rows=('job_id', 'size'),
    unique_jobs=('job_id', 'nunique'),
).sort_values('raw_rows', ascending=False)
display(job_distribution)
montreal_jobs = df_raw.loc[df_raw['job_city'] == 'Montreal']
montreal_unique_posting = montreal_jobs.drop_duplicates(subset=['job_title', 'employer_name', 'job_city'])
print('Unique job ids:', montreal_jobs['job_id'].nunique())
print('Unique title-employer-city combinations:', len(montreal_unique_posting))
montreal_employer = montreal_unique_posting.value_counts('employer_name').reset_index(name='count')
display(montreal_employer)
montreal_query = montreal_jobs.groupby(['role_family', 'search_level'], as_index=False).agg(
    raw_rows=('job_id', 'size'),
    unique_jobs=('job_id', 'nunique'),
)
display(montreal_query)
unique_jobs = df_raw.drop_duplicates('job_id')
montreal_share = unique_jobs['job_city'].eq('Montreal').mean()
print(f'Montreal share: {(montreal_share * 100).round(2)}%')

,raw_rows,unique_jobs
job_city,,
Montreal,273,157
NaN,218,204
Toronto,73,65
Toronto On,23,20
Markham,10,10
Cañada,7,7
Montréal,6,4
Mississauga,5,5
Québec City,4,4


Unique job ids: 157
Unique title-employer-city combinations: 151


,employer_name,count
0,Intact,9
1,TD,4
2,IBM,4
3,Pomerleau,3
4,Agoda,3
...,...,...
101,Hunter Bond,1
102,L3HHCM20,1
103,"LGS, une Société IBM / an IBM Company",1
104,Capgemini Sogeti,1


,role_family,search_level,raw_rows,unique_jobs
0,bi_analyst,entry_level,18,18
1,bi_analyst,internship,15,15
2,bi_analyst,new_grad,18,18
3,business_analyst,entry_level,24,24
4,business_analyst,internship,18,18
5,business_analyst,new_grad,17,17
6,data_analyst,entry_level,18,18
7,data_analyst,internship,18,18
8,data_analyst,new_grad,18,18
9,data_engineer,entry_level,18,18


Montreal share: 31.4%


In [15]:
# Check employment, remote, and salary
display(df_raw.value_counts('job_employment_type', dropna=False).reset_index(name='count'))
display(df_raw.value_counts('job_is_remote', dropna=False).reset_index(name='count'))
salary_word_pattern = r'\b(?:salary|pay|compensation|wage|hourly|per hour|annual|salaire|rémunération|CA\$|CAD\s*\$?|C\$|\$\s*\d)\b'
df_salary = df_raw.drop_duplicates('job_id').copy()
df_salary['mention_salary'] = df_salary['job_description'].str.contains(salary_word_pattern, case=False)
print('Job mention salary:', df_salary['mention_salary'].sum())
print(f'Job mention share: {(df_salary['mention_salary'].mean() * 100).round(2)}%')

salary_amount_pattern = r'((?:CA\$|C\$|CAD\s*\$?|\$)\s*\d[\d,]*(?:\.\d+)?\s*[kK]?)'
df_salary['salary_amount_found'] = df_salary['job_description'].str.extract(salary_amount_pattern, expand=False)
df_salary['has_salary_amount'] = df_salary['salary_amount_found'].notna()
print('Jobs with a possible salary amount:',df_salary['has_salary_amount'].sum())
print(f'Possible salary amount coverage: {(df_salary['has_salary_amount'].mean() * 100).round(2)}%')
salary_examples = df_salary.loc[df_salary['has_salary_amount'],['job_title', 'employer_name', 'job_city', 'salary_amount_found']]

display(salary_examples.head(20))

,job_employment_type,count
0,Full-time,410
1,Internship,116
2,Contract,54
3,NaN,36
4,Part-time,19
5,Contractor,6
6,Volunteer,1
7,Full-time and Internship,1


,job_is_remote,count
0,False,578
1,True,65


Job mention salary: 296
Job mention share: 59.2%
Jobs with a possible salary amount: 160
Possible salary amount coverage: 32.0%


,job_title,employer_name,job_city,salary_amount_found
10,"Stagiaire En Communications, Data Hub – Automn...",McKesson,Montreal,$16.56
11,BMO Capital Markets Winter 2027 Investment Ban...,BMO,Montreal,$1.5
13,National Real Estate Group Intern / Co-Op (Win...,TD,Montreal,"$45,700"
14,TDEF Credit Analyst Intern/Co-Op (Winter 2027),TD,Montreal,"$45,700"
15,TDEF Credit Analyst Intern/Co-Op (Winter 2027),TD,Montreal,"$45,700"
16,National Real Estate Group Intern / Co-Op (Win...,TD,Montreal,"$45,700"
17,Fall 2026 Co-Op: Internal Communications Inter...,CGI,Montreal,$18
18,TD Asset Management Portfolio Analytics Data A...,TD,Toronto,"$48,000"
19,"Data Science Risk Analyst, Winter 2027 (Co-Op/...",BMO Capital Markets,Toronto,"$54,600.00"
21,"Business Analyst, Data And Change Management ,...",BMO,Toronto,"$45,500.00"


In [16]:
# The summary for raw audit
audit_summary = pd.Series({
    'raw_rows': len(df_raw),
    'unique_jobs': df_raw['job_id'].nunique(),
    'duplicated_rows': df_raw['job_id'].duplicated().sum(),
    'missing_job_ids': df_raw['job_id'].isnull().sum(),
    'missing_titles': df_raw['job_title'].isnull().sum(),
    'missing_descriptions': df_raw['job_description'].isnull().sum(),
    'short_descriptions': len(short_jobs),
    'senior_title_roles': len(senior_rows),
    'invalid_posted_dates': posted_at.isna().sum(),
    'possible_salary_coverage': df_salary['has_salary_amount'].mean(),
}).rename_axis('info').reset_index()
display(audit_summary)

,info,0
0,raw_rows,643.00
1,unique_jobs,500.00
2,duplicated_rows,143.00
3,missing_job_ids,0.00
4,missing_titles,0.00
5,missing_descriptions,0.00
6,short_descriptions,35.00
7,senior_title_roles,79.00
8,invalid_posted_dates,12.00
9,possible_salary_coverage,0.32


## Raw Audit Findings
### 1. Dataset overview
- The raw dataset contains 643 records.
- There are 500 unique API job IDs, so 143 rows are duplicate occurrences of those jobs.
- Duplicates are expected because the same job may match multiple role-family and search-level queries.
- There are no missing job IDs, job titles, or job descriptions.
### 2. Data quality issues
- The `job_posted_at_datetime_utc` and `collected_at_utc` fields are still stored as strings and should be converted to UTC datetimes.
- `job_city` has 218 missing records, `job_employment_type` has 36 missing records, and `job_state` has 146 missing records.
- There are 35 short job descriptions with fewer than 500 characters.
- There are 79 potential senior positions based on title keywords.
- There are 49 postings older than 35 days.
- City and province names are not standardized; examples include `ON` and `Ontario`.
- There are 19 California records, which are outside the intended Canadian job-market scope.
- Both latitude and longitude are missing for 456 records.
- One volunteer position was returned and may not belong in the target employment market.
### Salary availability
- The structured salary fields have no usable coverage in this collection.
- Among the 500 unique job IDs, 296 postings (59.20%) mention salary-related language.
- 160 postings (32.00%) contain text that may represent a salary amount.
- Salary amounts extracted from job descriptions still require validation because some may refer to bonuses, allowances, benefits, or other non-salary amounts.
### Geographic limitations
- The collected data is strongly concentrated in Montreal.
- At least 157 unique job IDs use the exact city label `Montreal`, representing 31.40% of unique jobs.
- This geographic imbalance may reflect the API's search ranking and query coverage rather than the actual distribution of Canadian job opportunities.
- Therefore, the dataset cannot be treated as a representative random sample of the entire Canadian job market.
### Planned cleaning steps
1. Convert posting and collection timestamps to UTC datetime values.
2. Deduplicate the 643 raw records to 500 unique jobs using `job_id`.
3. Review the 35 short job descriptions and remove records that do not contain enough information for classification.
4. Standardize city and province names, and flag invalid or unknown locations. The 19 California records will be excluded from the Canadian dataset.
5. Standardize employment types, retain contract positions, label missing values as `unknown`, and exclude the volunteer position.
6. Reclassify role families and career levels using job titles and descriptions rather than relying only on the original search categories.
7. Review the 79 potential senior positions and exclude jobs confirmed to be outside the early-career scope.
8. Validate the 49 postings older than 35 days and flag confirmed outdated records.
9. Retain salary-related text for later extraction and validation.
10. Save the cleaned dataset and an exclusion report containing the reason each removed record was excluded.
### Overall conclusion
The raw data is sufficient for developing and demonstrating the job-market data pipeline, but it requires substantial deduplication, geographic normalization, and job classification before analysis. Conclusions from this dataset should be described as findings from the collected JSearch sample rather than estimates of the entire Canadian job market.